# 01 · Comprensión del negocio y de los datos: cifras del planteamiento

**Proyecto Final Integrador · Machine Learning I · Universidad EAN**
Integrantes: Claudia Naranjo, Yery Pedraza y Nicolás Ruiz.

Este cuaderno calcula, una por una, las cifras que se citan en
[`anteproyecto_PFI.md`](anteproyecto_PFI.md). Cada sección explica qué se mide, con qué datos y cómo.
Al final hay una tabla resumen con todos los indicadores.

**Confidencialidad.** El cuaderno solo muestra cifras agregadas: nunca imprime filas de
clientes ni de facturas.

**Montos.** Los importes están en pesos colombianos (COP) **escalados**: en la anonimización,
cada saldo se multiplicó por un factor secreto entre 0,85 y 1,15 (promedio 1). Un monto
individual puede diferir hasta un 15 % del real, pero al sumar miles de saldos los errores se
compensan, por lo que los totales son una buena aproximación.

## 0. Configuración

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

# Raíz del proyecto: la primera carpeta hacia arriba que contiene datos_crudos/ (ruta relativa,
# para que el cuaderno corra en el computador de cualquier integrante).
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datos_crudos").is_dir())
DATOS = RAIZ / "datos_crudos"

FECHA_CORTE = pd.Timestamp("2026-10-18")
UMBRAL_ATRASO = 30
TOL = 0.01  # saldos entre -0,01 y 0,01 se consideran en cero

print("Python", sys.version.split()[0], "| pandas", pd.__version__,
      "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)

Python 3.14.5 | pandas 3.0.6 | numpy 2.5.3 | scikit-learn 1.9.1


In [2]:
def leer(nombre):
    df = pd.read_csv(DATOS / nombre, encoding="utf-8-sig", low_memory=False)
    derivadas = {"fecha_anio", "fecha_mes", "fecha_trimestre", "fecha_dia_mes",
                 "fecha_dia_semana", "fecha_es_fin_de_mes"}
    for c in df.columns:
        if c.startswith("fecha") and c not in derivadas:
            df[c] = pd.to_datetime(df[c])
    return df

saldos = leer("saldos_anonimizados.csv")
movs = leer("movimientos_anonimizados.csv")
print("saldos:", saldos.shape, "| movimientos:", movs.shape)

saldos: (19262, 54) | movimientos: (40519, 44)


## 1. Qué contienen los datos

- **Saldo:** el registro contable de una cuenta por cobrar. Casi siempre nace de una factura,
  pero también puede nacer de una nota crédito, un ajuste o un saldo inicial.
- **Movimiento:** cada cargo o abono aplicado a un saldo: la factura que lo crea, los recibos de
  caja (`RC`) que lo pagan y las notas o ajustes que lo modifican.
- **Factura:** un saldo cuyo documento de origen es una factura electrónica (`FEV` o `FVE`); en
  los datos, la columna `es_deuda_facturada = True`. Es la deuda que nace de vender, y por eso
  es la unidad que se analiza.

In [3]:
resumen = pd.Series({
    "Saldos (filas de la tabla de saldos)": len(saldos),
    "Movimientos (filas de la tabla de movimientos)": len(movs),
    "Facturas (saldos con es_deuda_facturada)": int(saldos.es_deuda_facturada.sum()),
    "Clientes (nit_cliente distintos)": saldos.nit_cliente.nunique(),
    "Empresas del grupo": saldos.empresa.nunique(),
    "Unidades de negocio": saldos.unidad_negocio.nunique(),
    "Vendedores (nombre_vendedor distintos)": saldos.nombre_vendedor.nunique(),
    "Primera fecha de saldo": saldos.fecha.min().date(),
    "Última fecha de saldo": saldos.fecha.max().date(),
    "Fecha de corte de la extracción": saldos.fecha_corte.max().date(),
})
resumen.to_frame("valor")

,valor
Saldos (filas de la tabla de saldos),19262
Movimientos (filas de la tabla de movimientos),40519
Facturas (saldos con es_deuda_facturada),17920
Clientes (nit_cliente distintos),648
Empresas del grupo,4
Unidades de negocio,11
Vendedores (nombre_vendedor distintos),22
Primera fecha de saldo,2019-12-31
Última fecha de saldo,2026-10-16
Fecha de corte de la extracción,2026-10-18


In [4]:
print("Llave id_saldo única:", saldos.id_saldo.is_unique)
print("Movimientos sin saldo:", (~movs.id_saldo.isin(saldos.id_saldo)).sum())
print("Saldos sin movimientos:", (~saldos.id_saldo.isin(movs.id_saldo)).sum())
(saldos.empresa.value_counts(normalize=True) * 100).round(1).to_frame("% de saldos")

Llave id_saldo única: True
Movimientos sin saldo: 0
Saldos sin movimientos: 0


,% de saldos
empresa,
Empresa A,89.3
Empresa B,5.6
Empresa C,4.3
Empresa D,0.8


In [5]:
# Origen de los saldos: cuántos son facturas y cuántos otros documentos
origen = saldos.tipo_documento_origen.where(saldos.es_deuda_facturada, "Otros (FV, notas, ajustes, saldos iniciales…)")
origen.value_counts().to_frame("saldos")

,saldos
tipo_documento_origen,
FEV,17764
"Otros (FV, notas, ajustes, saldos iniciales…)",1342
FVE,156


## 2. Cuánto se factura y qué tan rápido se cobra (2025)

Se toman las facturas cuya `fecha` (emisión) cae en 2025, el último año completo.

- **Facturado en 2025:** suma de `total_debitos` de esas facturas. **Por día:** ese total ÷ 365.
- **Plazo pactado ponderado por monto:** promedio de `plazo_dias_observado` (vencimiento − emisión)
  donde cada factura pesa según su monto. Así, una factura de $100 M pesa 100 veces más que una de
  $1 M, porque para la caja importa dónde está el dinero, no cuántos documentos hay.
- **Días reales hasta cancelar, ponderado por monto:** igual, con `dias_hasta_cancelacion`
  (fecha en que el saldo quedó en cero − emisión). Solo cuentan las facturas ya canceladas.
- **Brecha:** días reales − plazo pactado = cuántos días después de lo acordado entra el dinero.

In [6]:
fac = saldos[saldos.es_deuda_facturada].copy()
f25 = fac[fac.fecha.dt.year == 2025]

facturado_2025 = f25.total_debitos.sum()
facturado_dia = facturado_2025 / 365
plazo_pond = np.average(f25.plazo_dias_observado, weights=f25.total_debitos)
canc = f25.dropna(subset=["dias_hasta_cancelacion"])
dias_pond = np.average(canc.dias_hasta_cancelacion, weights=canc.total_debitos)

pd.Series({
    "Facturas emitidas en 2025": len(f25),
    "Facturado 2025 (millones COP)": round(facturado_2025 / 1e6, 1),
    "Facturado por día (millones COP)": round(facturado_dia / 1e6, 1),
    "Plazo pactado ponderado (días)": round(plazo_pond, 1),
    "Días reales hasta cancelar, ponderado (días)": round(dias_pond, 1),
    "Brecha (días después de lo pactado)": round(dias_pond - plazo_pond, 1),
}).to_frame("valor")

,valor
Facturas emitidas en 2025,3344.0
Facturado 2025 (millones COP),23820.7
Facturado por día (millones COP),65.3
Plazo pactado ponderado (días),21.1
"Días reales hasta cancelar, ponderado (días)",43.4
Brecha (días después de lo pactado),22.3


**¿Por qué "cada día equivale a unos $65 M"?** La cartera que la empresa tiene por cobrar en un
momento dado es, en promedio, lo que factura en un día multiplicado por los días que tarda en
cobrar (es la idea del indicador **DSO**, *days sales outstanding*, o días de cartera). Si la
empresa cobrara un día antes, tendría por cobrar aproximadamente un día de facturación menos, es
decir, unos $65 M más en caja (capital de trabajo liberado). Cinco días menos ≈ 5 × $65 M.

In [7]:
pd.Series({f"{d} días menos de recaudo (millones COP)": round(d * facturado_dia / 1e6, 0)
           for d in [1, 5, 10]}).to_frame("caja liberada aprox.")

,caja liberada aprox.
1 días menos de recaudo (millones COP),65.0
5 días menos de recaudo (millones COP),326.0
10 días menos de recaudo (millones COP),653.0


## 3. Atraso de las facturas ya pagadas

**Población:** facturas **cerradas** (`estado_vencimiento_actual = SIN_SALDO`, saldo en cero) con
plazo no negativo. Un plazo negativo (vencimiento anterior a la emisión) es un error de registro y
se excluye.

**Atraso** = `fecha_cancelacion_registrada − fecha_vencimiento`, en días. Negativo: se pagó antes
de vencer; 0: el día del vencimiento; positivo: después. Se mide hasta que el saldo quedó en cero,
no hasta el primer abono, porque para la caja importa cuándo entra **todo** el dinero.

In [8]:
cerr = fac[(fac.estado_vencimiento_actual == "SIN_SALDO") & (fac.plazo_dias_observado >= 0)].copy()
cerr["atraso"] = (cerr.fecha_cancelacion_registrada - cerr.fecha_vencimiento).dt.days
cerr = cerr.dropna(subset=["atraso"])
print("Facturas cerradas con atraso medible:", len(cerr))
cerr.atraso.describe(percentiles=[.25, .5, .75, .9]).round(1).to_frame("atraso (días)")

Facturas cerradas con atraso medible: 17209


,atraso (días)
count,17209.0
mean,33.9
std,65.7
min,-62.0
25%,2.0
50%,16.0
75%,41.0
90%,84.0
max,1319.0


In [9]:
filas = []
for u in [0, 15, 30, 45, 60, 90]:
    x = cerr.atraso > u
    filas.append({"pagadas más de N días después del vencimiento": f"> {u}",
                  "% de facturas": round(x.mean() * 100, 1),
                  "% del monto": round(cerr.loc[x, "total_debitos"].sum() / cerr.total_debitos.sum() * 100, 1)})
pd.DataFrame(filas)

,pagadas más de N días después del vencimiento,% de facturas,% del monto
0,> 0,77.3,69.9
1,> 15,51.1,45.0
2,> 30,34.1,29.7
3,> 45,21.8,20.0
4,> 60,16.1,14.6
5,> 90,9.0,8.3


**Historia suficiente por cliente.** Para describir cómo paga un cliente se necesitan varias
facturas pagadas. Se cuenta cuántos clientes tienen al menos 5.

In [10]:
fact_por_cliente = cerr.groupby("nit_cliente").size()
pd.Series({
    "Clientes con al menos una factura pagada": len(fact_por_cliente),
    "Clientes con 5 o más facturas pagadas": int((fact_por_cliente >= 5).sum()),
    "% de las facturas pagadas que pertenecen a esos clientes": round(fact_por_cliente[fact_por_cliente >= 5].sum() / len(cerr) * 100, 1),
}).to_frame("valor")

,valor
Clientes con al menos una factura pagada,606.0
Clientes con 5 o más facturas pagadas,329.0
% de las facturas pagadas que pertenecen a esos clientes,96.7


**Por año de emisión.** Porcentaje de las facturas cerradas emitidas cada año que se pagaron
**más de 30 días** después de su vencimiento. Permite ver si el comportamiento es estable.

In [11]:
cerr["anio"] = cerr.fecha.dt.year
por_anio = cerr.groupby("anio").agg(facturas=("atraso", "size"),
                                    pct_mas_30=("atraso", lambda a: round((a > UMBRAL_ATRASO).mean() * 100, 1)))
por_anio

,facturas,pct_mas_30
anio,,
2020,969,43.0
2021,2104,42.8
2022,2522,33.5
2023,2879,31.3
2024,3134,31.9
2025,3318,33.4
2026,2283,30.7


In [12]:
r20_21 = (cerr.loc[cerr.anio.between(2020, 2021), "atraso"] > 30).mean() * 100
r22 = (cerr.loc[cerr.anio >= 2022, "atraso"] > 30).mean() * 100
print(f"2020-2021: {r20_21:.1f} % | 2022 en adelante: {r22:.1f} %")

2020-2021: 42.9 % | 2022 en adelante: 32.2 %


## 4. Cartera pendiente a la fecha de corte

Facturas con saldo mayor que $0,01 el 2026-10-18. **Días de mora** = fecha de corte − fecha de
vencimiento (0 si aún no vence). Los tramos siguen el reporte de cartera por edades.

In [13]:
pend = fac[fac.saldo_neto_actual > TOL].copy()
pend["mora"] = pend.dias_mora_actual.fillna(0)
tramos = pd.cut(pend.mora, [-1, 0, 30, 60, 90, 180, 360, np.inf],
                labels=["No vencida o 0", "1-30", "31-60", "61-90", "91-180", "181-360", "> 360"])
tabla = pend.groupby(tramos, observed=False).agg(facturas=("id_saldo", "size"),
                                                 monto_millones=("saldo_neto_actual", lambda s: round(s.sum() / 1e6, 1)))
print("Total pendiente (millones COP):", round(pend.saldo_neto_actual.sum() / 1e6, 1),
      "| clientes con saldo pendiente:", pend.nit_cliente.nunique())
print("Más de 90 días de mora (millones COP):", round(pend.loc[pend.mora > 90, "saldo_neto_actual"].sum() / 1e6, 1))
tabla

Total pendiente (millones COP): 3383.8 | clientes con saldo pendiente: 207
Más de 90 días de mora (millones COP): 537.9


,facturas,monto_millones
mora,,
No vencida o 0,117,1027.3
1-30,226,1091.4
31-60,127,496.6
61-90,52,230.7
91-180,89,344.6
181-360,69,175.5
> 360,4,17.7


## 5. ¿Cómo terminan los saldos?

- **% de saldos cerrados:** saldos (de cualquier origen) cuyo saldo actual está en cero.
- **Cierre con notas o ajustes:** entre las facturas cerradas, el último documento que las
  acreditó (`tipo_documento_cierre`) no fue un recibo de caja (`RC`). Es decir, la deuda se
  "cerró" sin entrada de dinero (descuento, anulación, compensación o ajuste contable).

In [14]:
fac_cerr = fac[fac.estado_vencimiento_actual == "SIN_SALDO"]
pd.Series({
    "% de saldos cerrados (todos los orígenes)": round((saldos.estado_vencimiento_actual == "SIN_SALDO").mean() * 100, 1),
    "% de facturas cerradas con un documento distinto de RC": round((fac_cerr.tipo_documento_cierre != "RC").mean() * 100, 1),
    "Saldos en la cuenta 131080 (difícil cobro)": int((saldos.codigo_cuenta == 131080).sum()),
}).to_frame("valor")

,valor
% de saldos cerrados (todos los orígenes),96.3
% de facturas cerradas con un documento distinto de RC,10.5
Saldos en la cuenta 131080 (difícil cobro),37.0


## 6. Carga diaria del equipo de cartera (reconstruida para 2025)

Los datos son una foto al corte, pero las fechas permiten **reconstruir** cómo estaba la cartera
cualquier día pasado. Para cada día hábil de 2025 (lunes a viernes) se cuenta:

- **Factura abierta ese día:** emitida en o antes de ese día y cancelada después (o aún no cancelada).
- **Factura vencida ese día:** abierta y con vencimiento anterior a ese día.
- **Más de 30 días de mora:** vencida hace más de 30 días.

Luego se resume con la mediana de los 261 días hábiles (el valor de un día típico).

In [15]:
fin = fac.fecha_cancelacion_registrada.fillna(pd.Timestamp("2100-01-01"))
filas = []
for t in pd.bdate_range("2025-01-01", "2025-12-31"):
    abierta = (fac.fecha <= t) & (fin > t)
    vencida = abierta & (fac.fecha_vencimiento < t)
    filas.append({"facturas_abiertas": abierta.sum(),
                  "clientes_con_saldo_abierto": fac.loc[abierta, "nit_cliente"].nunique(),
                  "clientes_con_factura_vencida": fac.loc[vencida, "nit_cliente"].nunique(),
                  "facturas_mas_30_dias_mora": (vencida & ((t - fac.fecha_vencimiento).dt.days > 30)).sum()})
carga = pd.DataFrame(filas)
carga.describe(percentiles=[.25, .5, .75]).loc[["count", "25%", "50%", "75%"]].round(0)

,facturas_abiertas,clientes_con_saldo_abierto,clientes_con_factura_vencida,facturas_mas_30_dias_mora
count,261.0,261.0,261.0,261.0
25%,415.0,171.0,125.0,157.0
50%,464.0,183.0,135.0,171.0
75%,512.0,193.0,145.0,187.0


In [16]:
dias_habiles = len(carga)
rc25 = movs[movs.es_recaudo & (movs.fecha_movimiento.dt.year == 2025)]
pd.Series({
    "Facturas nuevas por día hábil": round(len(f25) / dias_habiles, 1),
    "Saldos pagados con RC por día hábil (recaudos por aplicar)": round(rc25.id_saldo.nunique() / dias_habiles, 1),
}).to_frame("promedio 2025")

,promedio 2025
Facturas nuevas por día hábil,12.8
Saldos pagados con RC por día hábil (recaudos por aplicar),11.9


## 7. Hallazgos de calidad de datos

Cada hallazgo se contrasta con lo que dice el diccionario de datos (`datos_crudos/diccionario-datos.md`, fuera de Git por confidencialidad).

In [17]:
pd.Series({
    "Saldos con plazo negativo (vencimiento antes de la emisión)": int((saldos.plazo_dias_observado < 0).sum()),
    "Valores distintos de cuota (el diccionario dice 'normalmente 1')": sorted(saldos.cuota.unique().tolist()),
    "% con fecha_radicacion igual a fecha": round((saldos.fecha_radicacion == saldos.fecha).mean() * 100, 1),
    "Cuentas contables presentes": sorted(saldos.codigo_cuenta.unique().tolist()),
    "Tipos de cliente presentes": sorted(saldos.tipo_cliente.unique().tolist()),
    "% de saldos de clientes JURIDICA_NACIONAL": round((saldos.tipo_cliente == "JURIDICA_NACIONAL").mean() * 100, 1),
}).to_frame("valor")

,valor
Saldos con plazo negativo (vencimiento antes de la emisión),245
Valores distintos de cuota (el diccionario dice 'normalmente 1'),[0]
% con fecha_radicacion igual a fecha,99.7
Cuentas contables presentes,"[131010, 131015, 131080]"
Tipos de cliente presentes,"[EXTERIOR, JURIDICA_NACIONAL, NATURAL]"
% de saldos de clientes JURIDICA_NACIONAL,98.3


## 8. Resumen de cifras citadas en el planteamiento

In [18]:
pd.DataFrame([
    ("Facturado 2025 (millones COP)", round(facturado_2025 / 1e6, 0), "Σ total_debitos de facturas emitidas en 2025"),
    ("Facturado por día (millones COP)", round(facturado_dia / 1e6, 0), "Facturado 2025 ÷ 365"),
    ("Plazo pactado ponderado 2025 (días)", round(plazo_pond, 1), "Promedio de plazo ponderado por monto"),
    ("Días reales hasta cancelar 2025 (días)", round(dias_pond, 1), "Promedio de días hasta cancelación ponderado por monto"),
    ("% facturas cerradas pagadas después del vencimiento", round((cerr.atraso > 0).mean() * 100, 1), "Facturas cerradas con atraso > 0"),
    ("% facturas cerradas pagadas > 30 días tarde", round((cerr.atraso > 30).mean() * 100, 1), "Facturas cerradas con atraso > 30"),
    ("Cartera pendiente al corte (millones COP)", round(pend.saldo_neto_actual.sum() / 1e6, 0), "Σ saldo de facturas con saldo > 0,01"),
    ("Clientes con saldo pendiente", pend.nit_cliente.nunique(), "Clientes distintos en la cartera pendiente"),
    ("Cartera con > 90 días de mora (millones COP)", round(pend.loc[pend.mora > 90, "saldo_neto_actual"].sum() / 1e6, 0), "Σ saldo con mora > 90"),
    ("Clientes con factura vencida, día típico 2025", int(carga.clientes_con_factura_vencida.median()), "Mediana diaria reconstruida"),
], columns=["indicador", "valor", "cómo se calcula"])

,indicador,valor,cómo se calcula
0,Facturado 2025 (millones COP),23821.0,Σ total_debitos de facturas emitidas en 2025
1,Facturado por día (millones COP),65.0,Facturado 2025 ÷ 365
2,Plazo pactado ponderado 2025 (días),21.1,Promedio de plazo ponderado por monto
3,Días reales hasta cancelar 2025 (días),43.4,Promedio de días hasta cancelación ponderado p...
4,% facturas cerradas pagadas después del vencim...,77.3,Facturas cerradas con atraso > 0
5,% facturas cerradas pagadas > 30 días tarde,34.1,Facturas cerradas con atraso > 30
6,Cartera pendiente al corte (millones COP),3384.0,"Σ saldo de facturas con saldo > 0,01"
7,Clientes con saldo pendiente,207.0,Clientes distintos en la cartera pendiente
8,Cartera con > 90 días de mora (millones COP),538.0,Σ saldo con mora > 90
9,"Clientes con factura vencida, día típico 2025",135.0,Mediana diaria reconstruida
